# Hands-on Exercise 2 — Package & Run an MLflow Project
### AI Operations (AIOps) — MLflow Deep Dive | ~10–15 minutes

**Referenced in:** *MLflow Deep Dive Slide Deck*, Section 2 (MLflow Projects)

**Objective:** turn a training script into a reusable **MLflow Project** with a declared entry point,
parameters, and environment, then run it with the `mlflow run` CLI.

**Steps (from the slide deck):**
1. Turn your Exercise 1 training script into `train.py` that accepts CLI arguments (`argparse`).
2. Write an `MLproject` file with a `main` entry point and at least 2 parameters.
3. Add a `python_env.yaml` pinning your key library versions.
4. Run it with: `mlflow run . -P n_estimators=150 -P max_depth=6`
5. Confirm a new run appears in the MLflow UI, logged automatically by the Project run.
6. Bonus: re-run with a different parameter value and compare both runs in the UI.

**Deliverable:** a working MLflow Project directory (`MLproject` + `python_env.yaml` + `train.py`)
and a screenshot of the resulting run in the MLflow UI.

> **Prerequisite:** the MLflow Tracking Server from Exercise 1 must still be running at
> `http://localhost:5000`.

## Step 0 — Setup

In [ ]:
# !pip install mlflow scikit-learn pandas --quiet
import os, subprocess, sys
import mlflow

mlflow.set_tracking_uri("http://localhost:5000")
PROJECT_DIR = "iris_project"
os.makedirs(PROJECT_DIR, exist_ok=True)
print("Project directory:", os.path.abspath(PROJECT_DIR))

## Step 1 — Write `train.py` (accepts CLI arguments)
We use the `%%writefile` magic to save this cell's contents directly as a file inside the project directory.

In [ ]:
%%writefile iris_project/train.py
import argparse
import mlflow
import mlflow.sklearn
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--n_estimators", type=int, default=100)
    parser.add_argument("--max_depth", type=int, default=5)
    parser.add_argument("--data_path", type=str, default="")  # unused placeholder, mirrors slide example
    args = parser.parse_args()

    mlflow.set_tracking_uri("http://localhost:5000")
    # NOTE: we deliberately do NOT call mlflow.set_experiment() here.
    # `mlflow run` already creates and activates a run inside the experiment
    # you pass via `--experiment-name` on the command line (Step 4 below).
    # Calling set_experiment() here would point at a DIFFERENT experiment than
    # the one `mlflow run` already activated, and mlflow.start_run() would error.

    X, y = load_iris(return_X_y=True, as_frame=True)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    with mlflow.start_run(run_name=f"project-run-n{args.n_estimators}-d{args.max_depth}"):
        mlflow.log_param("n_estimators", args.n_estimators)
        mlflow.log_param("max_depth", args.max_depth)

        model = RandomForestClassifier(
            n_estimators=args.n_estimators, max_depth=args.max_depth, random_state=42
        )
        model.fit(X_train, y_train)
        preds = model.predict(X_test)

        acc = accuracy_score(y_test, preds)
        f1 = f1_score(y_test, preds, average="macro")
        mlflow.log_metric("accuracy", acc)
        mlflow.log_metric("f1_macro", f1)
        mlflow.sklearn.log_model(model, name="model")

        print(f"accuracy={acc:.4f}  f1_macro={f1:.4f}  run_id={mlflow.active_run().info.run_id}")


if __name__ == "__main__":
    main()

## Step 2 — Write the `MLproject` file
Declares the entry point and its parameters, matching the slide-deck example.

In [ ]:
%%writefile iris_project/MLproject
name: iris-classifier

python_env: python_env.yaml

entry_points:
  main:
    parameters:
      n_estimators: {type: int, default: 100}
      max_depth: {type: int, default: 5}
    command: "python train.py --n_estimators {n_estimators} --max_depth {max_depth}"

## Step 3 — Write `python_env.yaml`
Pins the exact interpreter and key dependencies, the Project-level analogue of Module 1's `environment.yml`.

In [ ]:
%%writefile iris_project/python_env.yaml
python: "3.12"
build_dependencies:
  - pip
  - setuptools
dependencies:
  - scikit-learn
  - pandas
  - mlflow

In [ ]:
# Confirm all three files exist
for fname in ["MLproject", "python_env.yaml", "train.py"]:
    path = os.path.join(PROJECT_DIR, fname)
    print(f"{'✓' if os.path.exists(path) else '✗'}  {path}")

## Step 4 — Run the project with `mlflow run`
This is the same command shown in the slide deck, plus `--experiment-name` so the run lands in the right experiment (see the note in `train.py` above).

In [ ]:
!mlflow run iris_project -P n_estimators=150 -P max_depth=6 \
    --experiment-name iris-classifier --env-manager=local

*(If your environment doesn't have the `mlflow` CLI on PATH inside the notebook kernel, run the same command in a terminal from this notebook's directory instead.)*

## Step 5 — Confirm the run appears in MLflow

In [ ]:
import mlflow
runs_df = mlflow.search_runs(
    experiment_names=["iris-classifier"],
    order_by=["start_time DESC"],
)
cols = [c for c in runs_df.columns if c in (
    "run_id", "tags.mlflow.runName", "params.n_estimators", "params.max_depth", "metrics.accuracy"
)]
print(runs_df[cols].head(3).to_string(index=False))

## Step 6 — Bonus: re-run with a different parameter value

In [ ]:
!mlflow run iris_project -P n_estimators=300 -P max_depth=10 \
    --experiment-name iris-classifier --env-manager=local

---
### ✅ Deliverable checklist
- [ ] `iris_project/MLproject` exists and defines a `main` entry point with ≥ 2 parameters
- [ ] `iris_project/python_env.yaml` pins your key library versions
- [ ] `iris_project/train.py` accepts `--n_estimators` and `--max_depth` via `argparse`
- [ ] At least one successful `mlflow run` visible as a new run in the MLflow UI
- [ ] Screenshot of the resulting run(s) in the MLflow UI